# 03 - Trimodal Model Training

This notebook trains the `TrimodalFusionModel` (Audio + Video + Text) with staged training.

**Architecture**:
- Audio: MFCC(120) + eGeMAPS(88) → AudioEncoder → (B, 64)
- Video: OpenFace(49) + CNN embed(512) → VideoEncoder → (B, 128)
- Text: SBERT(384) → StatsPool → (B, 768)
- Fusion: Cross-Modal Attention + Modality Gates → PHQ-8 score (regression) or depression probability (classification)

**Training modes**:
1. **Regression** (num_classes=1): Predicts PHQ-8 score (0-24), stages: text-only → audio+text → full trimodal
2. **Classification** (num_classes=2): Predicts depressed/not depressed, uses WeightedBCE loss + F1/ROC-AUC metrics

**Expected files per participant** in `data/raw/<PID>/`:
- `<PID>_OpenSMILE2.3.0_egemaps.csv` (88-dim)
- `<PID>_OpenSMILE2.3.0_mfcc.csv` (39 or 120-dim)
- `<PID>_BoVW_openFace_2.1.0_Pose_Gaze_AUs.csv` (49-dim)
- `<PID>_CNN_ResNet.csv` (512-dim)
- `<PID>_Transcript.csv` (text utterances)

In [ ]:
import sys
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.multimodal_fusion_v2 import TrimodalFusionModel, FusionOutput
from src.dataset.trimodal_dataset import (
    TrimodalDataset, trimodal_collate_fn,
    build_participant_data,
)
from src.training.losses import WeightedMSELoss, WeightedBCELoss
from src.training.metrics import compute_all_metrics, ClassificationMetricsTracker
from src.training.early_stopping import EarlyStopping

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 3.1 Load configuration

In [ ]:
config_path = Path("../configs/trimodal_v2_config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Configuration:")
for section, values in config.items():
    print(f"  {section}: {values}")

## 3.2 Load features from raw participant directories

Features are auto-loaded from `data/raw/<PID>/` by scanning for the expected CSV files.
Uses `build_participant_data()` from `trimodal_dataset` module.

In [ ]:
labels_csv = Path("../data/labels.csv")
raw_dir = Path("../data/raw")

labels_df = pd.read_csv(labels_csv)
print(f"Total labels: {len(labels_df)}")

# Build participant_data dict by scanning raw directories
participant_data = build_participant_data(str(raw_dir), labels_df, id_col="Participant_ID")

# Get train/dev splits if available, else use all data
train_csv = Path("../data/splits/train_split.csv")
dev_csv = Path("../data/splits/dev_split.csv")

if train_csv.exists() and dev_csv.exists():
    train_df = pd.read_csv(train_csv)
    dev_df = pd.read_csv(dev_csv)
    train_ids = [str(int(row["Participant_ID"])) for _, row in train_df.iterrows()]
    dev_ids = [str(int(row["Participant_ID"])) for _, row in dev_df.iterrows()]
    # Filter to IDs that have features loaded
    train_ids = [pid for pid in train_ids if pid in participant_data]
    dev_ids = [pid for pid in dev_ids if pid in participant_data]
else:
    all_ids = list(participant_data.keys())
    np.random.seed(42)
    np.random.shuffle(all_ids)
    n_train = int(0.7 * len(all_ids))
    n_dev = int(0.15 * len(all_ids))
    train_ids = all_ids[:n_train]
    dev_ids = all_ids[n_train:n_train + n_dev]

print(f"Train patients: {len(train_ids)}")
print(f"Dev patients: {len(dev_ids)}")
print(f"Feature keys per patient: {list(next(iter(participant_data.values())).keys())}")

## 3.3 Build DataLoaders

In [ ]:
## 3.3 Build DataLoaders

batch_size = config['data']['batch_size']
labels_dict = {
    str(int(row["Participant_ID"])): float(row["PHQ_Score"])
    for _, row in labels_df.iterrows()
}

train_dataset = TrimodalDataset(
    participant_data=participant_data,
    labels=labels_dict,
    participant_ids=train_ids,
    augment=True,
    temporal_dropout_rate=0.1,
    feature_noise_std=0.005,
    label_noise_std=0.15,
    depression_threshold=10.0,
    modality_dropout_prob=0.1,
)

dev_dataset = TrimodalDataset(
    participant_data=participant_data,
    labels=labels_dict,
    participant_ids=dev_ids,
    augment=False,
)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    collate_fn=trimodal_collate_fn, num_workers=0, pin_memory=True
)

dev_loader = DataLoader(
    dev_dataset, batch_size=batch_size, shuffle=False,
    collate_fn=trimodal_collate_fn, num_workers=0, pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Dev batches: {len(dev_loader)}")

## 3.4 Initialize Model

In [ ]:
## 3.4 Initialize Model

# Set num_classes=1 for regression (PHQ-8 score), num_classes=2 for binary classification
num_classes = config['model'].get('num_classes', 1)
task_type = "regression" if num_classes == 1 else "classification"

model = TrimodalFusionModel(
    fusion_dim=config['model']['fusion_dim'],
    num_attention_heads=config['model']['num_attention_heads'],
    stats_mode=config['model']['stats_mode'],
    dropout=config['model']['dropout'],
    modality_dropout=config['model']['modality_dropout'],
    openface_dim=config['features']['openface_dim'],
    cnn_dim=config['features']['cnn_dim'],
    num_classes=num_classes,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Task: {task_type}, num_classes={num_classes}")
print(f"Total params: {total_params:,}")
print(f"Trainable: {trainable_params:,}")

## 3.5 Training helper functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    n_batches = 0
    
    for batch in loader:
        kwargs = {}
        if batch['has_audio'].any():
            kwargs['mfcc'] = batch['audio_features'][..., :120].to(device)
            kwargs['egemaps'] = batch['audio_features'][..., 120:].to(device)
            kwargs['audio_mask'] = batch['audio_mask'].to(device)
        
        if batch['has_video'].any():
            kwargs['openface'] = batch['video_openface'].to(device)
            kwargs['cnn_embed'] = batch['video_cnn'].to(device)
            kwargs['video_mask'] = batch['video_mask'].to(device)
        
        if batch['has_text'].any():
            kwargs['text'] = batch['text_features'].to(device)
            kwargs['text_mask'] = batch['text_mask'].to(device)
        
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        output: FusionOutput = model(**kwargs)
        loss = criterion(output.prediction.squeeze(), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / max(n_batches, 1)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0
    n_batches = 0
    
    for batch in loader:
        kwargs = {}
        if batch['has_audio'].any():
            kwargs['mfcc'] = batch['audio_features'][..., :120].to(device)
            kwargs['egemaps'] = batch['audio_features'][..., 120:].to(device)
            kwargs['audio_mask'] = batch['audio_mask'].to(device)
        
        if batch['has_video'].any():
            kwargs['openface'] = batch['video_openface'].to(device)
            kwargs['cnn_embed'] = batch['video_cnn'].to(device)
            kwargs['video_mask'] = batch['video_mask'].to(device)
        
        if batch['has_text'].any():
            kwargs['text'] = batch['text_features'].to(device)
            kwargs['text_mask'] = batch['text_mask'].to(device)
        
        labels = batch['labels'].to(device)
        
        output: FusionOutput = model(**kwargs)
        preds = output.prediction.squeeze()
        loss = criterion(preds, labels)
        
        all_preds.append(preds.clamp(0, 24).cpu().numpy())
        all_targets.append(labels.cpu().numpy())
        total_loss += loss.item()
        n_batches += 1
    
    if n_batches == 0:
        return {'ccc': 0.0, 'rmse': 99.0, 'mae': 99.0, 'loss': 0.0}
    
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    metrics = compute_all_metrics(all_preds, all_targets)
    metrics['loss'] = total_loss / max(n_batches, 1)
    return metrics

print("Training functions defined.")

## 3.6 Stage 1: Text-only training (freeze audio + video)

In [ ]:
criterion = WeightedMSELoss(phq_threshold=10.0, high_weight=2.0, low_weight=1.0).to(device)

# Freeze audio and video
for name, param in model.named_parameters():
    if 'audio' in name or 'video' in name:
        param.requires_grad = False

s1_cfg = config['stages']['stage1']
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=s1_cfg['lr'], weight_decay=s1_cfg['weight_decay']
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=30)
early_stop = EarlyStopping(patience=s1_cfg['patience'], min_delta=0.001, mode='max')

best_s1 = {'ccc': -1.0, 'epoch': 0}

for epoch in range(1, s1_cfg['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_metrics = validate(model, dev_loader, criterion)
    scheduler.step(val_metrics['ccc'])
    
    if val_metrics['ccc'] > best_s1['ccc']:
        best_s1 = val_metrics.copy()
        best_s1['epoch'] = epoch
        torch.save(model.state_dict(), '../checkpoints/stage1_text.pt')
    
    if epoch % 10 == 0 or early_stop(val_metrics['ccc'], epoch):
        print(f"[S1 E{epoch:03d}] loss={train_loss:.4f} | val CCC={val_metrics['ccc']:.4f} | "
              f"RMSE={val_metrics['rmse']:.3f} | MAE={val_metrics['mae']:.3f}")
    
    if early_stop.is_stopped:
        break

print(f"\nStage 1 Best: CCC={best_s1['ccc']:.4f} at epoch {best_s1['epoch']}")

## 3.7 Stage 2: Audio + Text (freeze video, unfreeze audio)

In [ ]:
# Unfreeze audio, keep video frozen
for name, param in model.named_parameters():
    if 'video' in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

s2_cfg = config['stages']['stage2']
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=s2_cfg['lr'], weight_decay=s2_cfg['weight_decay']
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=20)
early_stop = EarlyStopping(patience=s2_cfg['patience'], min_delta=0.001, mode='max')

best_s2 = {'ccc': -1.0, 'epoch': 0}

for epoch in range(1, s2_cfg['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_metrics = validate(model, dev_loader, criterion)
    scheduler.step(val_metrics['ccc'])
    
    if val_metrics['ccc'] > best_s2['ccc']:
        best_s2 = val_metrics.copy()
        best_s2['epoch'] = epoch
        torch.save(model.state_dict(), '../checkpoints/stage2_audio_text.pt')
    
    if epoch % 10 == 0 or early_stop(val_metrics['ccc'], epoch):
        print(f"[S2 E{epoch:03d}] loss={train_loss:.4f} | val CCC={val_metrics['ccc']:.4f} | "
              f"RMSE={val_metrics['rmse']:.3f} | MAE={val_metrics['mae']:.3f}")
    
    if early_stop.is_stopped:
        break

print(f"\nStage 2 Best: CCC={best_s2['ccc']:.4f} at epoch {best_s2['epoch']}")

## 3.8 Stage 3: Full Trimodal (all unfrozen, reduced LR)

In [ ]:
# Unfreeze everything
for param in model.parameters():
    param.requires_grad = True

s3_cfg = config['stages']['stage3']
optimizer = torch.optim.AdamW(model.parameters(), lr=s3_cfg['lr'], weight_decay=s3_cfg['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=15)
early_stop = EarlyStopping(patience=s3_cfg['patience'], min_delta=0.001, mode='max')

best_s3 = {'ccc': -1.0, 'epoch': 0}

for epoch in range(1, s3_cfg['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_metrics = validate(model, dev_loader, criterion)
    scheduler.step(val_metrics['ccc'])
    
    if val_metrics['ccc'] > best_s3['ccc']:
        best_s3 = val_metrics.copy()
        best_s3['epoch'] = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'metrics': best_s3
        }, '../checkpoints/best_trimodal.pt')
    
    if epoch % 10 == 0 or early_stop(val_metrics['ccc'], epoch):
        print(f"[S3 E{epoch:03d}] loss={train_loss:.4f} | val CCC={val_metrics['ccc']:.4f} | "
              f"RMSE={val_metrics['rmse']:.3f} | MAE={val_metrics['mae']:.3f}")
    
    if early_stop.is_stopped:
        break

print(f"\nStage 3 Best: CCC={best_s3['ccc']:.4f} at epoch {best_s3['epoch']}")

## 3.9 Final summary

In [ ]:
print("=" * 50)
print("TRIMODAL TRAINING COMPLETE")
print(f"  Stage 1 (Text):       CCC={best_s1['ccc']:.4f}")
print(f"  Stage 2 (Audio+Text): CCC={best_s2['ccc']:.4f}")
print(f"  Stage 3 (Trimodal):   CCC={best_s3['ccc']:.4f}")
print(f"  Best checkpoint:      checkpoints/best_trimodal.pt")
print("=" * 50)

# --- Optional: Binary Classification Training ---
# To train a classifier instead, set num_classes=2 in the model init above
# and run this section:
if num_classes == 2:
    print("\n" + "=" * 50)
    print("BINARY CLASSIFICATION TRAINING")
    print("=" * 50)
    
    # Create binary labels
    binary_labels = {pid: 1 if score >= 10.0 else 0 for pid, score in labels_dict.items()}
    
    train_dataset_cls = TrimodalDataset(
        participant_data=participant_data,
        labels=binary_labels,
        participant_ids=train_ids,
        augment=True,
        depression_threshold=10.0,
    )
    dev_dataset_cls = TrimodalDataset(
        participant_data=participant_data,
        labels=binary_labels,
        participant_ids=dev_ids,
        augment=False,
    )
    
    train_loader_cls = DataLoader(train_dataset_cls, batch_size=batch_size, shuffle=True, collate_fn=trimodal_collate_fn)
    dev_loader_cls = DataLoader(dev_dataset_cls, batch_size=batch_size, shuffle=False, collate_fn=trimodal_collate_fn)
    
    # Use classification loss and metrics
    criterion_cls = WeightedBCELoss(pos_weight=None, focal_gamma=0.0, label_smoothing=0.1).to(device)
    metrics_tracker = ClassificationMetricsTracker()
    
    optimizer_cls = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler_cls = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cls, mode='max', factor=0.5, patience=10)
    
    print("Classification training loop ready. Run epochs with criterion_cls and metrics_tracker.")
    print("  Train script: scripts/train_trimodal_classifier.py")
    print("  Eval script:   scripts/evaluate_classifier.py")
    print("=" * 50)